# SupplyShield AI — WebShield Intelligence Engine

This notebook implements the WebShield layer of SupplyShield AI.

The engine evaluates publicly observable web signals to identify potentially
suspicious product/seller behaviour and generate explainable risk indicators.

Core signals:
- Price anomaly
- Seller/reputation risk
- Review anomaly
- Product similarity
- Listing completeness
- Availability inconsistency
- Counterfeit-risk proxy
- Overall WebShield risk

The system produces risk scores and explanations rather than claiming that
a product is definitively counterfeit or fraudulent.

In [1]:
# ============================================================
# CELL 2 — IMPORTS & ENVIRONMENT
# ============================================================

from __future__ import annotations

import os
import re
import json
import math
import logging
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("SupplyShield-WebShield")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If notebook is executed from member2/notebooks,
# move to project root automatically.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Explicit fallback for your current project structure
if not (PROJECT_ROOT / "member2").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "member2").exists():
        PROJECT_ROOT = candidate

DATA_DIR = PROJECT_ROOT / "member2" / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "member2" / "outputs" / "webshield"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_JSON = PROCESSED_DIR / "unified_supply_data.json"

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Processed directory: %s", PROCESSED_DIR)
logger.info("WebShield output directory: %s", OUTPUT_DIR)

2026-08-21 10:09:35,825 | INFO | Project root: c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI
2026-08-21 10:09:35,826 | INFO | Processed directory: c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed
2026-08-21 10:09:35,827 | INFO | WebShield output directory: c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield


In [2]:
# ============================================================
# CELL 3 — LOAD UPSTREAM DATASET
# ============================================================

def load_json_dataset(path: Path) -> pd.DataFrame:
    """
    Load a JSON dataset containing either:
    - a list of dictionaries
    - a dictionary containing records
    - a single dictionary record
    """

    if not path.exists():
        raise FileNotFoundError(
            f"\nInput dataset not found:\n{path}\n\n"
            "Run Notebook 02/04/05 first and verify that "
            "unified_supply_data.json exists."
        )

    with open(path, "r", encoding="utf-8") as file:
        payload = json.load(file)

    if isinstance(payload, list):
        records = payload

    elif isinstance(payload, dict):
        # Common possible container keys
        candidate_keys = [
            "records",
            "data",
            "items",
            "results",
            "products"
        ]

        records = None

        for key in candidate_keys:
            if key in payload and isinstance(payload[key], list):
                records = payload[key]
                break

        if records is None:
            records = [payload]

    else:
        raise ValueError(
            f"Unsupported JSON structure: {type(payload).__name__}"
        )

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError("The input dataset is empty.")

    return df


df = load_json_dataset(INPUT_JSON)

logger.info("Loaded dataset successfully.")
logger.info("Rows: %d", len(df))
logger.info("Columns: %d", len(df.columns))

print("=" * 70)
print("WEB SHIELD — INPUT DATASET")
print("=" * 70)
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")
print("\nAvailable columns:")
print(list(df.columns))

2026-08-21 10:09:52,909 | INFO | Loaded dataset successfully.
2026-08-21 10:09:52,910 | INFO | Rows: 303
2026-08-21 10:09:52,912 | INFO | Columns: 137


WEB SHIELD — INPUT DATASET
Rows    : 303
Columns : 137

Available columns:
['record_id', 'source', 'title', 'company', 'supplier', 'product', 'event', 'location', 'currency', 'url', 'availability_state', 'supplier_normalized', 'source_normalized', 'preliminary_risk_band', 'price', 'price_log', 'price_deviation_pct', 'price_robust_zscore', 'price_zscore', 'price_percentile', 'price_iqr_outlier', 'product_price_deviation_pct', 'supplier_price_deviation_pct', 'availability_risk_score', 'availability_text_length', 'availability_missing_flag', 'rating', 'rating_normalized', 'rating_risk_score', 'review_length', 'review_word_count', 'review_char_count', 'review_exclamation_count', 'review_question_count', 'review_uppercase_ratio', 'review_missing_flag', 'rating_missing_flag', 'review_quality_signal', 'disruption_keyword_count', 'negative_keyword_count', 'urgency_keyword_count', 'counterfeit_keyword_count', 'disruption_signal_flag', 'negative_signal_flag', 'urgency_signal_flag', 'counterfeit_

In [3]:

# ============================================================
# CELL 4 — SCHEMA NORMALIZATION
# ============================================================

def normalize_column_name(column: str) -> str:
    """
    Normalize column names into snake_case.
    """
    column = str(column).strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column)
    return column.strip("_")


df.columns = [normalize_column_name(col) for col in df.columns]

# ------------------------------------------------------------
# Canonical aliases
# ------------------------------------------------------------

COLUMN_ALIASES = {
    "name": "title",
    "product_name": "product",
    "product_title": "title",
    "seller": "supplier",
    "seller_name": "supplier",
    "vendor": "supplier",
    "company_name": "company",
    "review_count": "review",
    "reviews": "review",
    "availability_status": "availability",
    "product_url": "url",
    "link": "url"
}

for old_col, new_col in COLUMN_ALIASES.items():

    if old_col in df.columns and new_col not in df.columns:
        df[new_col] = df[old_col]

# ------------------------------------------------------------
# Ensure core WebShield fields exist
# ------------------------------------------------------------

CORE_COLUMNS = [
    "title",
    "product",
    "company",
    "supplier",
    "source",
    "price",
    "currency",
    "availability",
    "rating",
    "review",
    "url"
]

for column in CORE_COLUMNS:
    if column not in df.columns:
        df[column] = np.nan

logger.info("Schema normalization completed.")

print("Normalized columns:")
print(list(df.columns))

2026-08-21 10:20:08,756 | INFO | Schema normalization completed.


Normalized columns:
['record_id', 'source', 'title', 'company', 'supplier', 'product', 'event', 'location', 'currency', 'url', 'availability_state', 'supplier_normalized', 'source_normalized', 'preliminary_risk_band', 'price', 'price_log', 'price_deviation_pct', 'price_robust_zscore', 'price_zscore', 'price_percentile', 'price_iqr_outlier', 'product_price_deviation_pct', 'supplier_price_deviation_pct', 'availability_risk_score', 'availability_text_length', 'availability_missing_flag', 'rating', 'rating_normalized', 'rating_risk_score', 'review_length', 'review_word_count', 'review_char_count', 'review_exclamation_count', 'review_question_count', 'review_uppercase_ratio', 'review_missing_flag', 'rating_missing_flag', 'review_quality_signal', 'disruption_keyword_count', 'negative_keyword_count', 'urgency_keyword_count', 'counterfeit_keyword_count', 'disruption_signal_flag', 'negative_signal_flag', 'urgency_signal_flag', 'counterfeit_signal_flag', 'text_risk_score', 'supplier_observation_

In [4]:
# ============================================================
# CELL 5 — ROBUST FEATURE UTILITIES
# ============================================================

def safe_text(value: Any) -> str:
    """
    Convert arbitrary values to clean text.
    """
    if value is None:
        return ""

    if isinstance(value, float) and np.isnan(value):
        return ""

    return str(value).strip()


def safe_float(value: Any, default: float = np.nan) -> float:
    """
    Convert arbitrary values into float safely.
    """
    if value is None:
        return default

    if isinstance(value, (int, float, np.integer, np.floating)):
        if pd.isna(value):
            return default
        return float(value)

    text = safe_text(value)

    if not text:
        return default

    # Remove common currency symbols and separators
    text = text.replace(",", "")

    match = re.search(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if match:
        try:
            return float(match.group())
        except ValueError:
            return default

    return default


def extract_review_count(value: Any) -> float:
    """
    Extract review count from values such as:
        '131 reviews'
        '100 reviews 57 reviews'
        'No reviews'
    """
    text = safe_text(value).lower()

    if not text or "no review" in text:
        return 0.0

    numbers = re.findall(r"\d[\d,]*", text)

    if not numbers:
        return 0.0

    values = []

    for number in numbers:
        try:
            values.append(float(number.replace(",", "")))
        except ValueError:
            continue

    if not values:
        return 0.0

    # Use the largest observed count because scraped review strings
    # can contain repeated UI fragments.
    return max(values)


def normalize_rating(value: Any) -> float:
    """
    Convert rating to [0, 5].
    """
    rating = safe_float(value)

    if np.isnan(rating):
        return np.nan

    return float(np.clip(rating, 0, 5))


def minmax_series(series: pd.Series, default: float = 0.0) -> pd.Series:
    """
    Robust min-max normalization.
    """
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return pd.Series(
            np.full(len(series), default),
            index=series.index,
            dtype=float
        )

    minimum = numeric.min()
    maximum = numeric.max()

    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(
            np.full(len(series), default),
            index=series.index,
            dtype=float
        )

    return ((numeric - minimum) / (maximum - minimum)).clip(0, 1)


def sigmoid(x: float) -> float:
    """
    Numerically stable sigmoid.
    """
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))


logger.info("Utility functions initialized.")

2026-08-21 10:25:18,372 | INFO | Utility functions initialized.


In [5]:
# ============================================================
# CELL 6 — CANONICAL WEB DATA FEATURES
# ============================================================

# ------------------------------------------------------------
# Text fields
# ------------------------------------------------------------

TEXT_COLUMNS = [
    "title",
    "product",
    "company",
    "supplier",
    "source",
    "availability",
    "url"
]

for column in TEXT_COLUMNS:
    df[column] = df[column].apply(safe_text)

# ------------------------------------------------------------
# Numeric fields
# ------------------------------------------------------------

df["web_price"] = df["price"].apply(safe_float)
df["web_rating"] = df["rating"].apply(normalize_rating)
df["web_review_count"] = df["review"].apply(extract_review_count)

# ------------------------------------------------------------
# Combined product identity text
# ------------------------------------------------------------

df["web_product_text"] = (
    df["title"].fillna("")
    + " "
    + df["product"].fillna("")
    + " "
    + df["company"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()

# ------------------------------------------------------------
# Seller identity
# ------------------------------------------------------------

df["web_seller"] = (
    df["supplier"]
    .replace("", np.nan)
    .fillna(
        df["company"].replace("", np.nan)
    )
    .fillna(
        df["source"].replace("", np.nan)
    )
    .fillna("Unknown Seller")
)

# ------------------------------------------------------------
# Availability normalization
# ------------------------------------------------------------

def normalize_availability(value: Any) -> str:

    text = safe_text(value).lower()

    if not text:
        return "unknown"

    if any(token in text for token in [
        "out of stock",
        "unavailable",
        "sold out",
        "not available"
    ]):
        return "out_of_stock"

    if any(token in text for token in [
        "limited",
        "low stock",
        "few left"
    ]):
        return "limited"

    if any(token in text for token in [
        "in stock",
        "available",
        "ready"
    ]):
        return "available"

    return "unknown"


df["web_availability_status"] = (
    df["availability"]
    .apply(normalize_availability)
)

print("Canonical WebShield fields generated.")

display(
    df[
        [
            "web_product_text",
            "web_seller",
            "web_price",
            "web_rating",
            "web_review_count",
            "web_availability_status"
        ]
    ].head(10)
)

Canonical WebShield fields generated.


,web_product_text,web_seller,web_price,web_rating,web_review_count,web_availability_status
0,Wall Mount Mop Holder – No-Slide Grip for Home...,GlimmerHome,190.0,4.60,0.0,unknown
1,Compact TianMu Tool Set – Essential 9-Piece Re...,DeoDap,80.0,4.80,0.0,unknown
2,Manual Wall Fastening Nail Gun Tool Set (1 Set...,DeoDap,375.0,4.69,0.0,unknown
3,Mini Precision Screwdriver Set – Compact & Mul...,DeoDap,37.0,4.48,0.0,unknown
4,Electric Drill Machine – Compact & Powerful 28...,DeoDap,1.0,4.71,0.0,unknown
5,Leak Proof Tape – Instant Waterproof Seal for ...,BoltForce,102.0,3.89,0.0,unknown
6,Metal Try Square Ruler Set – Durable 2-Piece P...,DeoDap,136.0,4.71,0.0,unknown
7,Heavy Duty Sledge Hammer Rubber Mallet for Con...,DeoDap,165.0,4.73,0.0,unknown
8,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,BoltForce,207.0,4.23,0.0,unknown
9,Heavy Duty Electric Drill – Powerful & Versati...,DeoDap,1.0,4.48,0.0,unknown


In [6]:
# ============================================================
# CELL 7 — PRICE ANOMALY ENGINE
# ============================================================

def calculate_robust_price_anomaly(group: pd.Series) -> pd.Series:
    """
    Robust price anomaly using median and MAD.

    MAD is preferred over standard deviation because marketplace
    prices can contain extreme outliers.
    """

    prices = pd.to_numeric(group, errors="coerce")

    median = prices.median()

    if pd.isna(median) or median <= 0:
        return pd.Series(
            np.zeros(len(group)),
            index=group.index
        )

    deviations = (prices - median).abs()

    mad = deviations.median()

    if pd.isna(mad) or mad == 0:
        relative_deviation = (
            (prices - median).abs() / median
        )

        return relative_deviation.clip(0, 1)

    robust_z = (
        0.6745 * (prices - median) / mad
    ).abs()

    return (robust_z / 6.0).clip(0, 1)


# ------------------------------------------------------------
# Global price anomaly
# ------------------------------------------------------------

df["web_price_anomaly_global"] = calculate_robust_price_anomaly(
    df["web_price"]
)

# ------------------------------------------------------------
# Seller/product-group price anomaly
# ------------------------------------------------------------

group_keys = []

if df["web_seller"].nunique(dropna=True) > 1:
    group_keys.append("web_seller")

if df["company"].replace("", np.nan).notna().sum() > 0:
    group_keys.append("company")

if group_keys:

    grouped_price_anomaly = (
        df.groupby(group_keys, dropna=False)["web_price"]
        .transform(calculate_robust_price_anomaly)
    )

    df["web_price_anomaly_group"] = grouped_price_anomaly.fillna(0)

else:

    df["web_price_anomaly_group"] = 0.0


# ------------------------------------------------------------
# Combined price risk
# ------------------------------------------------------------

df["web_price_anomaly_score"] = (
    0.60 * df["web_price_anomaly_global"].fillna(0)
    + 0.40 * df["web_price_anomaly_group"].fillna(0)
).clip(0, 1)

logger.info("Price anomaly engine completed.")

print(
    df["web_price_anomaly_score"]
    .describe()
)

2026-08-21 10:26:03,040 | INFO | Price anomaly engine completed.


count    303.000000
mean       0.213636
std        0.245539
min        0.000000
25%        0.044428
50%        0.071893
75%        0.600000
max        0.644967
Name: web_price_anomaly_score, dtype: float64


In [7]:
# ============================================================
# CELL 8 — SELLER / REPUTATION RISK ENGINE
# ============================================================

# ------------------------------------------------------------
# Rating risk
# ------------------------------------------------------------

rating_filled = df["web_rating"].fillna(
    df["web_rating"].median()
    if df["web_rating"].notna().any()
    else 3.0
)

# Low rating -> higher risk
df["web_rating_risk"] = (
    (5.0 - rating_filled) / 5.0
).clip(0, 1)

# ------------------------------------------------------------
# Review volume risk
# ------------------------------------------------------------

review_count = df["web_review_count"].fillna(0)

if review_count.max() > 0:

    log_reviews = np.log1p(review_count)

    review_strength = minmax_series(
        pd.Series(log_reviews, index=df.index)
    )

    # Very low review history -> higher uncertainty/risk
    df["web_review_volume_risk"] = (
        1.0 - review_strength
    ).clip(0, 1)

else:

    df["web_review_volume_risk"] = 1.0


# ------------------------------------------------------------
# Seller identity completeness
# ------------------------------------------------------------

seller_missing = (
    df["web_seller"].eq("")
    | df["web_seller"].isna()
    | df["web_seller"].eq("Unknown Seller")
)

df["web_seller_identity_risk"] = seller_missing.astype(float)


# ------------------------------------------------------------
# URL presence
# ------------------------------------------------------------

df["web_url_missing_risk"] = (
    df["url"].eq("")
    | df["url"].isna()
).astype(float)


# ------------------------------------------------------------
# Seller reputation risk
# ------------------------------------------------------------

df["web_seller_reputation_score"] = (
    0.35 * df["web_rating_risk"]
    + 0.30 * df["web_review_volume_risk"]
    + 0.20 * df["web_seller_identity_risk"]
    + 0.15 * df["web_url_missing_risk"]
).clip(0, 1)

logger.info("Seller reputation engine completed.")

display(
    df[
        [
            "web_seller",
            "web_rating",
            "web_review_count",
            "web_rating_risk",
            "web_review_volume_risk",
            "web_seller_reputation_score"
        ]
    ].head(10)
)

2026-08-21 10:26:16,998 | INFO | Seller reputation engine completed.


,web_seller,web_rating,web_review_count,web_rating_risk,web_review_volume_risk,web_seller_reputation_score
0,GlimmerHome,4.60,0.0,0.080,1.0,0.3280
1,DeoDap,4.80,0.0,0.040,1.0,0.3140
2,DeoDap,4.69,0.0,0.062,1.0,0.3217
3,DeoDap,4.48,0.0,0.104,1.0,0.3364
4,DeoDap,4.71,0.0,0.058,1.0,0.3203
5,BoltForce,3.89,0.0,0.222,1.0,0.3777
6,DeoDap,4.71,0.0,0.058,1.0,0.3203
7,DeoDap,4.73,0.0,0.054,1.0,0.3189
8,BoltForce,4.23,0.0,0.154,1.0,0.3539
9,DeoDap,4.48,0.0,0.104,1.0,0.3364


In [8]:
# ============================================================
# CELL 9 — REVIEW ANOMALY ENGINE
# ============================================================

def review_text_features(value: Any) -> Dict[str, float]:

    text = safe_text(value).lower()

    if not text:
        return {
            "review_text_length": 0,
            "review_repetition_score": 0.0,
            "review_generic_score": 0.0
        }

    tokens = re.findall(r"\b[a-z]{2,}\b", text)

    if not tokens:
        return {
            "review_text_length": 0,
            "review_repetition_score": 0.0,
            "review_generic_score": 0.0
        }

    unique_ratio = len(set(tokens)) / max(len(tokens), 1)

    repetition_score = 1.0 - unique_ratio

    generic_terms = [
        "good",
        "nice",
        "best",
        "excellent",
        "amazing",
        "quality",
        "product",
        "worth",
        "awesome"
    ]

    generic_count = sum(
        token in generic_terms
        for token in tokens
    )

    generic_score = min(
        generic_count / max(len(tokens), 1) * 5,
        1.0
    )

    return {
        "review_text_length": len(tokens),
        "review_repetition_score": float(
            np.clip(repetition_score, 0, 1)
        ),
        "review_generic_score": float(
            np.clip(generic_score, 0, 1)
        )
    }


review_features = df["review"].apply(review_text_features)

review_features_df = pd.DataFrame(
    list(review_features),
    index=df.index
)

df = pd.concat(
    [df, review_features_df],
    axis=1
)

# ------------------------------------------------------------
# Combined review anomaly
# ------------------------------------------------------------

df["web_review_anomaly_score"] = (
    0.55 * df["review_repetition_score"]
    + 0.45 * df["review_generic_score"]
).clip(0, 1)

logger.info("Review anomaly engine completed.")

print(
    "Review anomaly statistics:"
)

print(
    df["web_review_anomaly_score"]
    .describe()
)

2026-08-21 10:26:30,964 | INFO | Review anomaly engine completed.


Review anomaly statistics:
count    303.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: web_review_anomaly_score, dtype: float64


In [9]:
# ============================================================
# CELL 10 — PRODUCT SIMILARITY ENGINE
# ============================================================

product_text = (
    df["web_product_text"]
    .fillna("")
    .replace("", "unknown product")
)

# ------------------------------------------------------------
# TF-IDF representation
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=5000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(product_text)

# ------------------------------------------------------------
# Pairwise similarity
# ------------------------------------------------------------

similarity_matrix = cosine_similarity(tfidf_matrix)

np.fill_diagonal(similarity_matrix, 0.0)

# ------------------------------------------------------------
# Maximum similarity to another listing
# ------------------------------------------------------------

if len(df) > 1:

    df["web_max_product_similarity"] = (
        similarity_matrix.max(axis=1)
    )

else:

    df["web_max_product_similarity"] = 0.0


# ------------------------------------------------------------
# Similarity should NOT automatically mean counterfeit.
# Instead, it is used as a supporting signal.
# ------------------------------------------------------------

df["web_product_similarity_risk"] = (
    df["web_max_product_similarity"]
    .clip(0, 1)
)

logger.info(
    "Product similarity engine completed using TF-IDF."
)

print(
    df[
        [
            "title",
            "web_max_product_similarity",
            "web_product_similarity_risk"
        ]
    ].head(10)
)

2026-08-21 10:26:58,829 | INFO | Product similarity engine completed using TF-IDF.


                                               title  \
0  Wall Mount Mop Holder – No-Slide Grip for Home...   
1  Compact TianMu Tool Set – Essential 9-Piece Re...   
2    Manual Wall Fastening Nail Gun Tool Set (1 Set)   
3  Mini Precision Screwdriver Set – Compact & Mul...   
4  Electric Drill Machine – Compact & Powerful 28...   
5  Leak Proof Tape – Instant Waterproof Seal for ...   
6  Metal Try Square Ruler Set – Durable 2-Piece P...   
7  Heavy Duty Sledge Hammer Rubber Mallet for Con...   
8  Hardware Tool Set – 11 Pcs Multi-Functional Ki...   
9  Heavy Duty Electric Drill – Powerful & Versati...   

   web_max_product_similarity  web_product_similarity_risk  
0                    0.098874                     0.098874  
1                    0.180759                     0.180759  
2                    0.634006                     0.634006  
3                    0.359280                     0.359280  
4                    0.290732                     0.290732  
5                

In [10]:
# ============================================================
# CELL 11 — LISTING QUALITY & COMPLETENESS SIGNAL
# ============================================================

# Required observable fields
QUALITY_FIELDS = [
    "title",
    "product",
    "company",
    "supplier",
    "price",
    "rating",
    "review",
    "url"
]

quality_matrix = pd.DataFrame(index=df.index)

for column in QUALITY_FIELDS:

    if column in ["price", "rating"]:

        quality_matrix[column] = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            ).notna().astype(float)
        )

    else:

        quality_matrix[column] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .astype(float)
        )

df["web_listing_completeness"] = (
    quality_matrix.mean(axis=1)
).clip(0, 1)

df["web_listing_quality_risk"] = (
    1.0 - df["web_listing_completeness"]
).clip(0, 1)

logger.info("Listing completeness analysis completed.")

print(
    df["web_listing_completeness"]
    .describe()
)

2026-08-21 10:27:15,176 | INFO | Listing completeness analysis completed.


count    303.00
mean       0.75
std        0.00
min        0.75
25%        0.75
50%        0.75
75%        0.75
max        0.75
Name: web_listing_completeness, dtype: float64


In [11]:
# ============================================================
# CELL 12 — AVAILABILITY / MARKET CONSISTENCY SIGNAL
# ============================================================

availability_map = {
    "available": 0.0,
    "limited": 0.45,
    "out_of_stock": 0.70,
    "unknown": 0.35
}

df["web_availability_risk"] = (
    df["web_availability_status"]
    .map(availability_map)
    .fillna(0.35)
    .clip(0, 1)
)

# ------------------------------------------------------------
# Price + availability interaction
# ------------------------------------------------------------

# Extremely high price anomaly combined with limited/out-of-stock
# availability can represent unusual market behaviour.

df["web_market_inconsistency_score"] = (
    0.65 * df["web_price_anomaly_score"]
    + 0.35 * df["web_availability_risk"]
).clip(0, 1)

logger.info("Availability consistency signal completed.")

2026-08-21 10:27:29,849 | INFO | Availability consistency signal completed.


In [12]:
# ============================================================
# CELL 13 — WEBSHIELD SIGNAL FUSION
# ============================================================

# ------------------------------------------------------------
# Individual signals
# ------------------------------------------------------------

SIGNAL_COLUMNS = [
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score"
]

for column in SIGNAL_COLUMNS:
    df[column] = (
        pd.to_numeric(
            df[column],
            errors="coerce"
        )
        .fillna(0)
        .clip(0, 1)
    )


# ------------------------------------------------------------
# WebShield risk weights
# ------------------------------------------------------------

WEBSHIELD_WEIGHTS = {
    "web_price_anomaly_score": 0.25,
    "web_seller_reputation_score": 0.25,
    "web_review_anomaly_score": 0.15,
    "web_product_similarity_risk": 0.10,
    "web_listing_quality_risk": 0.10,
    "web_market_inconsistency_score": 0.15
}

weight_sum = sum(WEBSHIELD_WEIGHTS.values())

if not math.isclose(weight_sum, 1.0, rel_tol=1e-9):

    WEBSHIELD_WEIGHTS = {
        key: value / weight_sum
        for key, value in WEBSHIELD_WEIGHTS.items()
    }


# ------------------------------------------------------------
# Weighted fusion
# ------------------------------------------------------------

df["webshield_risk_score"] = 0.0

for signal, weight in WEBSHIELD_WEIGHTS.items():

    df["webshield_risk_score"] += (
        df[signal] * weight
    )

df["webshield_risk_score"] = (
    df["webshield_risk_score"]
    .clip(0, 1)
)


# ------------------------------------------------------------
# Convert to 0–100
# ------------------------------------------------------------

df["webshield_risk_score_100"] = (
    df["webshield_risk_score"] * 100
).round(2)


logger.info("WebShield signal fusion completed.")

print(
    df["webshield_risk_score_100"]
    .describe()
)

2026-08-21 10:27:43,042 | INFO | WebShield signal fusion completed.


count    303.000000
mean      23.027327
std        8.780690
min       13.800000
25%       16.575000
50%       18.580000
75%       34.800000
max       41.630000
Name: webshield_risk_score_100, dtype: float64


In [13]:
# ============================================================
# CELL 14 — WEBSHIELD RISK BANDS
# ============================================================

def risk_band(score: float) -> str:

    score = safe_float(score, 0.0)

    if score >= 80:
        return "CRITICAL"

    if score >= 60:
        return "HIGH"

    if score >= 40:
        return "MEDIUM"

    if score >= 20:
        return "LOW"

    return "MINIMAL"


df["webshield_risk_band"] = (
    df["webshield_risk_score_100"]
    .apply(risk_band)
)

RISK_ORDER = {
    "MINIMAL": 0,
    "LOW": 1,
    "MEDIUM": 2,
    "HIGH": 3,
    "CRITICAL": 4
}

df["webshield_risk_priority"] = (
    df["webshield_risk_band"]
    .map(RISK_ORDER)
    .fillna(0)
    .astype(int)
)

print(
    df["webshield_risk_band"]
    .value_counts()
    .sort_index()
)

webshield_risk_band
LOW        115
MEDIUM       3
MINIMAL    185
Name: count, dtype: int64


In [14]:
# ============================================================
# CELL 15 — EXPLAINABLE WEBSHIELD RISK REASONS
# ============================================================

def generate_webshield_reasons(row: pd.Series) -> List[str]:

    reasons = []

    price_score = float(row.get(
        "web_price_anomaly_score", 0
    ))

    seller_score = float(row.get(
        "web_seller_reputation_score", 0
    ))

    review_score = float(row.get(
        "web_review_anomaly_score", 0
    ))

    similarity_score = float(row.get(
        "web_product_similarity_risk", 0
    ))

    quality_score = float(row.get(
        "web_listing_quality_risk", 0
    ))

    market_score = float(row.get(
        "web_market_inconsistency_score", 0
    ))

    availability = safe_text(
        row.get("web_availability_status", "")
    )

    # --------------------------------------------------------
    # Price
    # --------------------------------------------------------

    if price_score >= 0.75:
        reasons.append(
            "Strong price deviation detected"
        )

    elif price_score >= 0.50:
        reasons.append(
            "Moderate price anomaly detected"
        )

    # --------------------------------------------------------
    # Seller
    # --------------------------------------------------------

    if seller_score >= 0.75:
        reasons.append(
            "Elevated seller/reputation risk"
        )

    elif seller_score >= 0.50:
        reasons.append(
            "Limited or weak seller reputation signals"
        )

    # --------------------------------------------------------
    # Reviews
    # --------------------------------------------------------

    if review_score >= 0.75:
        reasons.append(
            "Potentially unusual review-pattern signal"
        )

    elif review_score >= 0.50:
        reasons.append(
            "Review-pattern anomaly detected"
        )

    # --------------------------------------------------------
    # Similarity
    # --------------------------------------------------------

    if similarity_score >= 0.85:
        reasons.append(
            "High similarity with another observed listing"
        )

    elif similarity_score >= 0.70:
        reasons.append(
            "Strong product-listing similarity detected"
        )

    # --------------------------------------------------------
    # Listing quality
    # --------------------------------------------------------

    if quality_score >= 0.75:
        reasons.append(
            "Listing has significant missing information"
        )

    elif quality_score >= 0.50:
        reasons.append(
            "Listing completeness is below expected level"
        )

    # --------------------------------------------------------
    # Market consistency
    # --------------------------------------------------------

    if market_score >= 0.75:
        reasons.append(
            "Strong market-consistency anomaly"
        )

    # --------------------------------------------------------
    # Availability
    # --------------------------------------------------------

    if availability == "out_of_stock":
        reasons.append(
            "Product currently appears unavailable"
        )

    elif availability == "limited":
        reasons.append(
            "Limited availability signal detected"
        )

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    if not reasons:
        reasons.append(
            "No strong WebShield anomaly detected"
        )

    return reasons


df["webshield_risk_reasons"] = (
    df.apply(
        generate_webshield_reasons,
        axis=1
    )
)

logger.info(
    "Explainable WebShield reasons generated."
)

display(
    df[
        [
            "title",
            "webshield_risk_score_100",
            "webshield_risk_band",
            "webshield_risk_reasons"
        ]
    ].head(10)
)

2026-08-21 10:28:06,959 | INFO | Explainable WebShield reasons generated.


,title,webshield_risk_score_100,webshield_risk_band,webshield_risk_reasons
0,Wall Mount Mop Holder – No-Slide Grip for Home...,15.66,MINIMAL,[No strong WebShield anomaly detected]
1,Compact TianMu Tool Set – Essential 9-Piece Re...,16.76,MINIMAL,[No strong WebShield anomaly detected]
2,Manual Wall Fastening Nail Gun Tool Set (1 Set),25.63,LOW,[No strong WebShield anomaly detected]
3,Mini Precision Screwdriver Set – Compact & Mul...,20.05,LOW,[No strong WebShield anomaly detected]
4,Electric Drill Machine – Compact & Powerful 28...,19.75,MINIMAL,[No strong WebShield anomaly detected]
5,Leak Proof Tape – Instant Waterproof Seal for ...,18.75,MINIMAL,[No strong WebShield anomaly detected]
6,Metal Try Square Ruler Set – Durable 2-Piece P...,16.54,MINIMAL,[No strong WebShield anomaly detected]
7,Heavy Duty Sledge Hammer Rubber Mallet for Con...,17.81,MINIMAL,[No strong WebShield anomaly detected]
8,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,18.87,MINIMAL,[No strong WebShield anomaly detected]
9,Heavy Duty Electric Drill – Powerful & Versati...,20.15,LOW,[No strong WebShield anomaly detected]


In [15]:
# ============================================================
# CELL 16 — COUNTERFEIT-RISK PROXY
# ============================================================

# IMPORTANT:
# This is a risk proxy, NOT a definitive counterfeit classifier.
#
# It combines observable web signals:
# - unusual pricing
# - seller reputation
# - review anomalies
# - product similarity
# - listing incompleteness
#
# A high score means the listing deserves investigation.

COUNTERFEIT_WEIGHTS = {
    "web_price_anomaly_score": 0.30,
    "web_seller_reputation_score": 0.25,
    "web_review_anomaly_score": 0.15,
    "web_product_similarity_risk": 0.15,
    "web_listing_quality_risk": 0.15
}

counterfeit_weight_sum = sum(
    COUNTERFEIT_WEIGHTS.values()
)

COUNTERFEIT_WEIGHTS = {
    key: value / counterfeit_weight_sum
    for key, value in COUNTERFEIT_WEIGHTS.items()
}

df["web_counterfeit_risk_score"] = 0.0

for signal, weight in COUNTERFEIT_WEIGHTS.items():

    df["web_counterfeit_risk_score"] += (
        df[signal] * weight
    )

df["web_counterfeit_risk_score"] = (
    df["web_counterfeit_risk_score"]
    .clip(0, 1)
)

df["web_counterfeit_risk_100"] = (
    df["web_counterfeit_risk_score"] * 100
).round(2)

df["web_counterfeit_risk_band"] = (
    df["web_counterfeit_risk_100"]
    .apply(risk_band)
)

logger.info(
    "Counterfeit-risk proxy generated."
)

display(
    df[
        [
            "title",
            "web_counterfeit_risk_100",
            "web_counterfeit_risk_band"
        ]
    ]
    .sort_values(
        "web_counterfeit_risk_100",
        ascending=False
    )
    .head(15)
)

2026-08-21 10:28:27,629 | INFO | Counterfeit-risk proxy generated.


,title,web_counterfeit_risk_100,web_counterfeit_risk_band
99,Siemens Artis Zee Cath Lab - Application: Hosp...,41.96,MEDIUM
33,Hydraulic Pallet Truck - Body Material: Steel,39.60,LOW
236,Hydraulic Pallet Truck - Color: Yellow,39.60,LOW
55,Garco Back Pressure Regulator - Application: I...,38.92,LOW
42,Broken Bag Detector - Accuracy: +2 %,38.87,LOW
69,Fruit Hardness Tester,38.53,LOW
176,Fruit Hardness Tester - Color: Silver,38.53,LOW
155,Silver Chloride,38.46,LOW
135,10mm Broken Bag Detector - Frequency: 50hz,38.19,LOW
256,Back Pressure Regulator Application: Industrial,38.01,LOW


In [16]:
# ============================================================
# CELL 17 — SUSPICIOUS LISTING DECISION ENGINE
# ============================================================

def classify_listing(row: pd.Series) -> str:

    webshield = safe_float(
        row.get("webshield_risk_score_100"),
        0
    )

    counterfeit = safe_float(
        row.get("web_counterfeit_risk_100"),
        0
    )

    # High-confidence review queue
    if (
        counterfeit >= 80
        or webshield >= 85
    ):
        return "REVIEW_REQUIRED"

    if (
        counterfeit >= 65
        or webshield >= 70
    ):
        return "WATCHLIST"

    if (
        counterfeit >= 45
        or webshield >= 50
    ):
        return "MONITOR"

    return "NORMAL"


df["webshield_action"] = (
    df.apply(
        classify_listing,
        axis=1
    )
)

logger.info(
    "Suspicious listing classification completed."
)

print(
    df["webshield_action"]
    .value_counts()
)

2026-08-21 10:28:44,016 | INFO | Suspicious listing classification completed.


webshield_action
NORMAL    303
Name: count, dtype: int64


In [17]:
# ============================================================
# CELL 18 — WEB SHIELD PRIORITY QUEUE
# ============================================================

# Combined operational priority.
#
# WebShield risk receives the highest weight because this notebook
# specifically represents the web-intelligence layer.

df["webshield_priority_score"] = (
    0.60 * df["webshield_risk_score"]
    + 0.40 * df["web_counterfeit_risk_score"]
).clip(0, 1)

df["webshield_priority_score_100"] = (
    df["webshield_priority_score"] * 100
).round(2)

priority_df = (
    df[
        [
            "web_seller",
            "title",
            "web_price",
            "web_rating",
            "web_review_count",
            "webshield_risk_score_100",
            "web_counterfeit_risk_100",
            "webshield_priority_score_100",
            "webshield_risk_band",
            "webshield_action",
            "url"
        ]
    ]
    .sort_values(
        "webshield_priority_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

priority_df.insert(
    0,
    "priority_rank",
    np.arange(1, len(priority_df) + 1)
)

print("=" * 90)
print("TOP WEBSHIELD PRIORITY LISTINGS")
print("=" * 90)

display(
    priority_df.head(20)
)

TOP WEBSHIELD PRIORITY LISTINGS


,priority_rank,web_seller,title,web_price,web_rating,web_review_count,webshield_risk_score_100,web_counterfeit_risk_100,webshield_priority_score_100,webshield_risk_band,webshield_action,url
0,1,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab - Application: Hosp...,14000000.0,4.2,0.0,41.63,41.96,41.76,MEDIUM,NORMAL,https://www.tradeindia.com/products/siemens-ar...
1,2,G M Industries,Hydraulic Pallet Truck - Body Material: Steel,26800.0,4.2,0.0,40.05,39.60,39.87,MEDIUM,NORMAL,https://www.tradeindia.com/products/hydraulic-...
2,3,Hardware City,Hydraulic Pallet Truck - Color: Yellow,14160.0,4.2,0.0,40.05,39.60,39.87,MEDIUM,NORMAL,https://www.tradeindia.com/products/hydraulic-...
3,4,Aditya Agency,Silver Chloride,60000.0,4.2,0.0,39.95,38.46,39.36,LOW,NORMAL,https://www.tradeindia.com/products/silver-chl...
4,5,App Pumps Engineering Company,Garco Back Pressure Regulator - Application: I...,200000.0,4.2,0.0,39.60,38.92,39.33,LOW,NORMAL,https://www.tradeindia.com/products/garco-back...
5,6,Applied Techno Engineers Private Limited,Broken Bag Detector - Accuracy: +2 %,58000.0,4.2,0.0,39.57,38.87,39.29,LOW,NORMAL,https://www.tradeindia.com/products/broken-bag...
6,7,Mxrady Lab Solutions Private Limited,Fruit Hardness Tester - Color: Silver,32450.0,4.2,0.0,39.34,38.53,39.02,LOW,NORMAL,https://www.tradeindia.com/products/fruit-hard...
7,8,Tamilnadu Engineering Instruments,Fruit Hardness Tester,30000.0,4.2,0.0,39.34,38.53,39.02,LOW,NORMAL,https://www.tradeindia.com/products/fruit-hard...
8,9,Soarmlich Engineers,10mm Broken Bag Detector - Frequency: 50hz,35231.0,4.2,0.0,39.12,38.19,38.75,LOW,NORMAL,https://www.tradeindia.com/products/10mm-broke...
9,10,Axis Solutions Limited,Back Pressure Regulator Application: Industrial,8000.0,4.2,0.0,38.54,38.01,38.33,LOW,NORMAL,https://www.tradeindia.com/products/back-press...


In [18]:
# ============================================================
# CELL 19 — WEBSHIELD OPERATIONAL SUMMARY
# ============================================================

summary = {
    "total_listings": int(len(df)),

    "critical_webshield": int(
        (df["webshield_risk_band"] == "CRITICAL").sum()
    ),

    "high_webshield": int(
        (df["webshield_risk_band"] == "HIGH").sum()
    ),

    "counterfeit_review_required": int(
        (
            df["web_counterfeit_risk_band"]
            .isin(["CRITICAL", "HIGH"])
        ).sum()
    ),

    "review_required": int(
        (df["webshield_action"] == "REVIEW_REQUIRED").sum()
    ),

    "watchlist": int(
        (df["webshield_action"] == "WATCHLIST").sum()
    ),

    "monitor": int(
        (df["webshield_action"] == "MONITOR").sum()
    ),

    "normal": int(
        (df["webshield_action"] == "NORMAL").sum()
    ),

    "mean_webshield_score": round(
        float(df["webshield_risk_score_100"].mean()),
        2
    ),

    "max_webshield_score": round(
        float(df["webshield_risk_score_100"].max()),
        2
    ),

    "mean_counterfeit_proxy": round(
        float(df["web_counterfeit_risk_100"].mean()),
        2
    ),

    "max_counterfeit_proxy": round(
        float(df["web_counterfeit_risk_100"].max()),
        2
    )
}

print("=" * 75)
print("SUPPLYSHIELD AI — WEBSHIELD SUMMARY")
print("=" * 75)

for key, value in summary.items():
    print(f"{key:32s}: {value}")

SUPPLYSHIELD AI — WEBSHIELD SUMMARY
total_listings                  : 303
critical_webshield              : 0
high_webshield                  : 0
counterfeit_review_required     : 0
review_required                 : 0
watchlist                       : 0
monitor                         : 0
normal                          : 303
mean_webshield_score            : 23.03
max_webshield_score             : 41.63
mean_counterfeit_proxy          : 22.61
max_counterfeit_proxy           : 41.96


In [19]:
# ============================================================
# CELL 20 — SIGNAL DIAGNOSTICS
# ============================================================

diagnostic_columns = [
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score",
    "webshield_risk_score",
    "web_counterfeit_risk_score"
]

diagnostics = []

for column in diagnostic_columns:

    series = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    diagnostics.append({
        "signal": column,
        "mean": round(series.mean(), 4),
        "median": round(series.median(), 4),
        "min": round(series.min(), 4),
        "max": round(series.max(), 4),
        "non_zero_count": int(
            (series > 0).sum()
        ),
        "missing_count": int(
            series.isna().sum()
        )
    })

diagnostics_df = pd.DataFrame(diagnostics)

display(diagnostics_df)

,signal,mean,median,min,max,non_zero_count,missing_count
0,web_price_anomaly_score,0.2136,0.0719,0.0000,0.6450,256,0
1,web_seller_reputation_score,0.3560,0.3560,0.3000,0.4330,303,0
2,web_review_anomaly_score,0.0000,0.0000,0.0000,0.0000,0,0
3,web_product_similarity_risk,0.2365,0.2090,0.0462,0.7539,303,0
4,web_listing_quality_risk,0.2500,0.2500,0.2500,0.2500,303,0
5,web_market_inconsistency_score,0.2614,0.1692,0.1225,0.5417,303,0
6,webshield_risk_score,0.2303,0.1858,0.1380,0.4163,303,0
7,web_counterfeit_risk_score,0.2261,0.1895,0.1350,0.4196,303,0


In [20]:
# ============================================================
# CELL 21 — WEBSHIELD SIGNAL CORRELATION
# ============================================================

correlation_columns = [
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score",
    "webshield_risk_score",
    "web_counterfeit_risk_score"
]

correlation_matrix = (
    df[correlation_columns]
    .corr()
    .round(3)
)

display(correlation_matrix)

,web_price_anomaly_score,web_seller_reputation_score,web_review_anomaly_score,web_product_similarity_risk,web_listing_quality_risk,web_market_inconsistency_score,webshield_risk_score,web_counterfeit_risk_score
web_price_anomaly_score,1.000,-0.033,NaN,0.095,NaN,1.000,0.986,0.961
web_seller_reputation_score,-0.033,1.000,NaN,-0.173,NaN,-0.033,-0.027,-0.042
web_review_anomaly_score,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
web_product_similarity_risk,0.095,-0.173,NaN,1.000,NaN,0.095,0.253,0.362
web_listing_quality_risk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
web_market_inconsistency_score,1.000,-0.033,NaN,0.095,NaN,1.000,0.986,0.961
webshield_risk_score,0.986,-0.027,NaN,0.253,NaN,0.986,1.000,0.993
web_counterfeit_risk_score,0.961,-0.042,NaN,0.362,NaN,0.961,0.993,1.000


In [21]:
# ============================================================
# CELL 22 — FINAL WEBSHIELD DATASET
# ============================================================

WEBSHIELD_OUTPUT_COLUMNS = [
    # Identity
    "source",
    "title",
    "product",
    "company",
    "supplier",
    "web_seller",
    "url",

    # Market information
    "web_price",
    "currency",
    "web_rating",
    "web_review_count",
    "web_availability_status",

    # WebShield signals
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score",

    # Main WebShield outputs
    "webshield_risk_score",
    "webshield_risk_score_100",
    "webshield_risk_band",

    # Counterfeit proxy
    "web_counterfeit_risk_score",
    "web_counterfeit_risk_100",
    "web_counterfeit_risk_band",

    # Operational decision
    "webshield_priority_score",
    "webshield_priority_score_100",
    "webshield_action",

    # Explainability
    "webshield_risk_reasons"
]

available_output_columns = [
    column
    for column in WEBSHIELD_OUTPUT_COLUMNS
    if column in df.columns
]

webshield_df = (
    df[available_output_columns]
    .copy()
)

logger.info(
    "Final WebShield dataframe shape: %s",
    webshield_df.shape
)

print(
    f"Final WebShield dataframe shape: "
    f"{webshield_df.shape}"
)

display(
    webshield_df.head(10)
)

2026-08-21 10:30:00,895 | INFO | Final WebShield dataframe shape: (303, 28)


Final WebShield dataframe shape: (303, 28)


,source,title,product,company,supplier,web_seller,url,web_price,currency,web_rating,...,webshield_risk_score,webshield_risk_score_100,webshield_risk_band,web_counterfeit_risk_score,web_counterfeit_risk_100,web_counterfeit_risk_band,webshield_priority_score,webshield_priority_score_100,webshield_action,webshield_risk_reasons
0,DeoDap,Wall Mount Mop Holder – No-Slide Grip for Home...,Wall Mount Mop Holder – No-Slide Grip for Home...,GlimmerHome,,GlimmerHome,https://deodap.in/products/hardware-tool-multi...,190.0,INR,4.60,...,0.156596,15.66,MINIMAL,0.152749,15.27,MINIMAL,0.155057,15.51,NORMAL,[No strong WebShield anomaly detected]
1,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Re...,Compact TianMu Tool Set – Essential 9-Piece Re...,DeoDap,,DeoDap,https://deodap.in/products/compact-tianmu-comb...,80.0,INR,4.80,...,0.167612,16.76,MINIMAL,0.166994,16.70,MINIMAL,0.167365,16.74,NORMAL,[No strong WebShield anomaly detected]
2,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),Manual Wall Fastening Nail Gun Tool Set (1 Set),DeoDap,,DeoDap,https://deodap.in/products/manual-wall-fasteni...,375.0,INR,4.69,...,0.256317,25.63,LOW,0.272695,27.27,LOW,0.262868,26.29,NORMAL,[No strong WebShield anomaly detected]
3,DeoDap,Mini Precision Screwdriver Set – Compact & Mul...,Mini Precision Screwdriver Set – Compact & Mul...,DeoDap,,DeoDap,https://deodap.in/products/mini-precision-scre...,37.0,INR,4.48,...,0.200465,20.05,LOW,0.207488,20.75,LOW,0.203274,20.33,NORMAL,[No strong WebShield anomaly detected]
4,DeoDap,Electric Drill Machine – Compact & Powerful 28...,Electric Drill Machine – Compact & Powerful 28...,DeoDap,,DeoDap,https://deodap.in/products/high-performance-el...,1.0,INR,4.71,...,0.197456,19.75,MINIMAL,0.199976,20.00,LOW,0.198464,19.85,NORMAL,[No strong WebShield anomaly detected]
5,DeoDap,Leak Proof Tape – Instant Waterproof Seal for ...,Leak Proof Tape – Instant Waterproof Seal for ...,BoltForce,,BoltForce,https://deodap.in/products/0405_leak_proof_tape-1,102.0,INR,3.89,...,0.187528,18.75,MINIMAL,0.181413,18.14,MINIMAL,0.185082,18.51,NORMAL,[No strong WebShield anomaly detected]
6,DeoDap,Metal Try Square Ruler Set – Durable 2-Piece P...,Metal Try Square Ruler Set – Durable 2-Piece P...,DeoDap,,DeoDap,https://deodap.in/products/heavy-duty-metal-tr...,136.0,INR,4.71,...,0.165395,16.54,MINIMAL,0.161213,16.12,MINIMAL,0.163723,16.37,NORMAL,[No strong WebShield anomaly detected]
7,DeoDap,Heavy Duty Sledge Hammer Rubber Mallet for Con...,Heavy Duty Sledge Hammer Rubber Mallet for Con...,DeoDap,,DeoDap,https://deodap.in/products/heavy-duty-sledge-h...,165.0,INR,4.73,...,0.178109,17.81,MINIMAL,0.177458,17.75,MINIMAL,0.177848,17.78,NORMAL,[No strong WebShield anomaly detected]
8,DeoDap,Hardware Tool Set – 11 Pcs Multi-Functional Ki...,Hardware Tool Set – 11 Pcs Multi-Functional Kit,BoltForce,,BoltForce,https://deodap.in/products/15800_multi_hardwar...,207.0,INR,4.23,...,0.188668,18.87,MINIMAL,0.187974,18.80,MINIMAL,0.188390,18.84,NORMAL,[No strong WebShield anomaly detected]
9,DeoDap,Heavy Duty Electric Drill – Powerful & Versati...,Heavy Duty Electric Drill – Powerful & Versati...,DeoDap,,DeoDap,https://deodap.in/products/heavy-duty-electric...,1.0,INR,4.48,...,0.201481,20.15,LOW,0.204001,20.40,LOW,0.202489,20.25,NORMAL,[No strong WebShield anomaly detected]


In [22]:
# ============================================================
# CELL 23 — EXPORT WEBSHIELD CSV
# ============================================================

CSV_OUTPUT = (
    OUTPUT_DIR /
    "webshield_risk_results.csv"
)

webshield_export = webshield_df.copy()

# Convert list-valued explanation column to readable text
if "webshield_risk_reasons" in webshield_export.columns:

    webshield_export["webshield_risk_reasons"] = (
        webshield_export["webshield_risk_reasons"]
        .apply(
            lambda values:
            " | ".join(values)
            if isinstance(values, list)
            else safe_text(values)
        )
    )

webshield_export.to_csv(
    CSV_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV exported successfully:\n{CSV_OUTPUT}")

CSV exported successfully:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_risk_results.csv


In [23]:
# ============================================================
# CELL 24 — EXPORT WEBSHIELD JSON
# ============================================================

JSON_OUTPUT = (
    OUTPUT_DIR /
    "webshield_risk_results.json"
)

def make_json_safe(value: Any) -> Any:

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, list):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in value.items()
        }

    if pd.isna(value):
        return None

    return value


records = []

for record in webshield_df.to_dict(
    orient="records"
):

    records.append(
        {
            key: make_json_safe(value)
            for key, value in record.items()
        }
    )

with open(
    JSON_OUTPUT,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        records,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"JSON exported successfully:\n{JSON_OUTPUT}")

JSON exported successfully:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_risk_results.json


In [24]:
# ============================================================
# CELL 25 — HIGH-RISK INVESTIGATION QUEUE
# ============================================================

investigation_queue = (
    webshield_export[
        webshield_export["webshield_action"]
        .isin(
            [
                "REVIEW_REQUIRED",
                "WATCHLIST"
            ]
        )
    ]
    .sort_values(
        "webshield_priority_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

QUEUE_OUTPUT = (
    OUTPUT_DIR /
    "webshield_investigation_queue.csv"
)

investigation_queue.to_csv(
    QUEUE_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 80)
print("WEBSHIELD INVESTIGATION QUEUE")
print("=" * 80)

print(
    f"Records requiring attention: "
    f"{len(investigation_queue):,}"
)

display(
    investigation_queue.head(25)
)

print(
    f"\nQueue exported to:\n{QUEUE_OUTPUT}"
)

WEBSHIELD INVESTIGATION QUEUE
Records requiring attention: 0


,source,title,product,company,supplier,web_seller,url,web_price,currency,web_rating,...,webshield_risk_score,webshield_risk_score_100,webshield_risk_band,web_counterfeit_risk_score,web_counterfeit_risk_100,web_counterfeit_risk_band,webshield_priority_score,webshield_priority_score_100,webshield_action,webshield_risk_reasons



Queue exported to:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_investigation_queue.csv


In [25]:
# ============================================================
# CELL 26 — FINAL WEBSHIELD VALIDATION
# ============================================================

REQUIRED_OUTPUTS = [
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score",
    "webshield_risk_score",
    "webshield_risk_score_100",
    "webshield_risk_band",
    "web_counterfeit_risk_score",
    "web_counterfeit_risk_100",
    "web_counterfeit_risk_band",
    "webshield_action"
]

missing_outputs = [
    column
    for column in REQUIRED_OUTPUTS
    if column not in df.columns
]

assert not missing_outputs, (
    f"Missing WebShield outputs: {missing_outputs}"
)

# ------------------------------------------------------------
# Range validation
# ------------------------------------------------------------

for column in [
    "web_price_anomaly_score",
    "web_seller_reputation_score",
    "web_review_anomaly_score",
    "web_product_similarity_risk",
    "web_listing_quality_risk",
    "web_market_inconsistency_score",
    "webshield_risk_score",
    "web_counterfeit_risk_score"
]:

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    ).dropna()

    assert (
        (values >= 0).all()
        and
        (values <= 1).all()
    ), f"{column} contains values outside [0,1]"


for column in [
    "webshield_risk_score_100",
    "web_counterfeit_risk_100"
]:

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    ).dropna()

    assert (
        (values >= 0).all()
        and
        (values <= 100).all()
    ), f"{column} contains values outside [0,100]"


# ------------------------------------------------------------
# Export validation
# ------------------------------------------------------------

assert CSV_OUTPUT.exists(), (
    "WebShield CSV was not created."
)

assert JSON_OUTPUT.exists(), (
    "WebShield JSON was not created."
)

print("=" * 75)
print("WEBSHIELD VALIDATION PASSED")
print("=" * 75)
print(f"Input records             : {len(df):,}")
print(f"Output records            : {len(webshield_df):,}")
print(f"WebShield features        : {len(REQUIRED_OUTPUTS)}")
print(f"CSV output                : {CSV_OUTPUT}")
print(f"JSON output               : {JSON_OUTPUT}")
print(f"Investigation queue       : {QUEUE_OUTPUT}")
print("=" * 75)

WEBSHIELD VALIDATION PASSED
Input records             : 303
Output records            : 303
WebShield features        : 13
CSV output                : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_risk_results.csv
JSON output               : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_risk_results.json
Investigation queue       : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\outputs\webshield\webshield_investigation_queue.csv


In [26]:
# ============================================================
# CELL 27 — EXECUTIVE WEBSHIELD OUTPUT
# ============================================================

executive_columns = [
    "web_seller",
    "title",
    "web_price",
    "web_rating",
    "web_review_count",
    "webshield_risk_score_100",
    "webshield_risk_band",
    "web_counterfeit_risk_100",
    "web_counterfeit_risk_band",
    "webshield_action"
]

executive_view = (
    df[executive_columns]
    .sort_values(
        [
            "webshield_risk_score_100",
            "web_counterfeit_risk_100"
        ],
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

print("=" * 100)
print("SUPPLYSHIELD AI — WEBSHIELD TOP RISK LISTINGS")
print("=" * 100)

display(executive_view)

print("\nWebShield engine completed successfully.")

SUPPLYSHIELD AI — WEBSHIELD TOP RISK LISTINGS


,web_seller,title,web_price,web_rating,web_review_count,webshield_risk_score_100,webshield_risk_band,web_counterfeit_risk_100,web_counterfeit_risk_band,webshield_action
0,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab - Application: Hosp...,14000000.0,4.2,0.0,41.63,MEDIUM,41.96,MEDIUM,NORMAL
1,G M Industries,Hydraulic Pallet Truck - Body Material: Steel,26800.0,4.2,0.0,40.05,MEDIUM,39.60,LOW,NORMAL
2,Hardware City,Hydraulic Pallet Truck - Color: Yellow,14160.0,4.2,0.0,40.05,MEDIUM,39.60,LOW,NORMAL
3,Aditya Agency,Silver Chloride,60000.0,4.2,0.0,39.95,LOW,38.46,LOW,NORMAL
4,App Pumps Engineering Company,Garco Back Pressure Regulator - Application: I...,200000.0,4.2,0.0,39.60,LOW,38.92,LOW,NORMAL
5,Applied Techno Engineers Private Limited,Broken Bag Detector - Accuracy: +2 %,58000.0,4.2,0.0,39.57,LOW,38.87,LOW,NORMAL
6,Tamilnadu Engineering Instruments,Fruit Hardness Tester,30000.0,4.2,0.0,39.34,LOW,38.53,LOW,NORMAL
7,Mxrady Lab Solutions Private Limited,Fruit Hardness Tester - Color: Silver,32450.0,4.2,0.0,39.34,LOW,38.53,LOW,NORMAL
8,Soarmlich Engineers,10mm Broken Bag Detector - Frequency: 50hz,35231.0,4.2,0.0,39.12,LOW,38.19,LOW,NORMAL
9,Fluid Centric Equipments & Services,Br2-high-pressure Back-pressure Regulator - Ap...,23600.0,4.2,0.0,38.59,LOW,37.41,LOW,NORMAL



WebShield engine completed successfully.
